# 07. 项目说明文件

AGDR VLA 入门教程 · 第 3 章 · 后半周

- **学习目标**：理解 `AGENTS.md` / `CLAUDE.md` 是怎样被 Agent 读取和使用的，知道里面该写什么、不该写什么，能为一个项目写出一份合格的项目说明文件
- **前置知识**：[03-上下文管理与压缩](./03-上下文管理与压缩.ipynb)、[05-Skill与Hook](./05-Skill与Hook.ipynb)、[06-写好提示词](./06-写好提示词.ipynb)
- **预计学习时长**：1 小时

## 项目说明文件是什么

上一节最后说到：如果你总是在对话里补充同样的信息，就该把它们写下来。**项目说明文件**就是写给 Agent 看的项目 README，放在项目根目录，Agent 每次启动时自动读取。

| 文件名 | 谁会读 |
| --- | --- |
| `AGENTS.md` | 一个开放的约定，Codex 等许多 Agent 工具都会读取 |
| `CLAUDE.md` | Claude Code 读取的文件 |

两者的作用完全相同，只是文件名不同。如果项目里的同学用的工具不一样，可以把内容写在 `AGENTS.md` 里，再让 `CLAUDE.md` 引用它。**本仓库就是这么做的**，打开根目录的 [CLAUDE.md](../CLAUDE.md) 看看，里面只有一个引用：

```markdown
@AGENTS.md
```

Claude Code 读到 `@文件路径` 时，会把那个文件的内容一起加载进来。这样两个文件只需要维护一份内容。

## 它是怎样起作用的

原理很简单：Agent 启动时读取这个文件，把内容放进**系统提示词**（或者会话最开头的消息）里。之后的每一次请求，模型都能看到它。

这正好解释了 03 节提到的一点：**项目说明文件的内容不会被上下文压缩丢掉。** 压缩替换的是对话历史，系统提示词每次都原样保留。

我们在 mini agent 里实现一遍，就清楚了。

In [ ]:
import mini_agent as ma

BASE_PROMPT = ma.SYSTEM_PROMPT
ma.setup_workspace()

# 在沙箱工作区里放一份项目说明文件
agents_md = "\n".join([
    "# 机械臂抓取实验",
    "",
    "## 项目说明",
    "- train.log 是训练日志，config.json 是超参数",
    "- 日志每行以 [INFO] / [WARN] / [ERROR] 开头",
    "",
    "## 回答规范",
    "- 涉及多个数值时，用表格呈现",
    "- 给出结论时，注明依据来自哪个文件",
    "",
    "## 约束",
    "- 不要修改 config.json：超参数要经过组会讨论才能改。需要调整时，只给出建议",
])
(ma.WORKSPACE / "AGENTS.md").write_text(agents_md + "\n", encoding="utf-8")
print("模式:", "在线" if ma.ONLINE else "离线（读取录制的模型响应）")
print(sorted(p.name for p in ma.WORKSPACE.iterdir()))

In [ ]:
def load_project_instructions(workspace):
    path = workspace / "AGENTS.md"
    if not path.exists():
        return ""
    return "\n\n以下是本项目的说明文件 AGENTS.md，请遵守其中的要求：\n\n" + path.read_text(encoding="utf-8")


ma.SYSTEM_PROMPT = BASE_PROMPT + load_project_instructions(ma.WORKSPACE)
print(ma.SYSTEM_PROMPT)

现在给 Agent 一个任务，**任务里不提任何格式要求，还故意要求它改配置**：

In [ ]:
answer, _ = ma.run_agent(
    "最低的 loss 是多少？顺便把学习率调小一点。",
    offline_script=["agentsmd_read.json", "agentsmd_answer.json"],
)
print("\n最终回答:\n")
print(answer)

回答用了表格、注明了依据的来源，并且没有修改 config.json，而是给出了建议。这三点都来自 `AGENTS.md`，用户的提问里一个字都没提。

注意「约束」那一条的写法：它不只写了「不要修改」，还写了**原因**（要经过组会讨论）和**替代做法**（只给出建议）。06 节说过，解释原因比加重语气更有效。

> 这里的约束仍然靠模型自觉遵守。如果「不能改某个文件」是必须保证的，还要配合 Hook 或权限设置（05、09 节）。

> 在线模式下，可以删掉 `AGENTS.md` 中的某一条，重新运行上面三个单元格，看看回答有什么变化。

## 案例：本仓库的 AGENTS.md

打开根目录的 [AGENTS.md](../AGENTS.md) 读一遍。你在学习本教程时，Agent 会自动进入「问答模式 / 答题模式」，拒绝替你运行命令，这些行为全部来自这一个文件。它的结构是：

| 部分 | 作用 |
| --- | --- |
| 项目背景 | 一段话说清这是什么项目、给谁用 |
| 如何判断角色 | 区分作者和学习者，决定采用哪套规则 |
| 通用规则 | 所有情况下都适用的规矩：用中文、不做破坏性操作、不擅自提交 |
| 学习者模式 | 最核心的部分：手动操作原则、问答模式、答题模式 |
| 作者模式 | 协助编写教程时的规则 |
| 仓库结构与约定 | 目录结构和命名规则 |

值得留意的一个设计：各章的 `Agent选择题协议.md` 并没有写进 `AGENTS.md`，而是让 Agent **在进入答题模式时才去读取**对应章节的协议。这和 05 节 Skill 的渐进式加载是同一个思路：每次都需要的规则放进 `AGENTS.md`，只在特定任务里需要的细节单独成文件，按需读取。

> 你可以在自己的 Agent 里问它：「根据 AGENTS.md，你现在处于什么模式？你不能替我做哪些事？」检查它是否真的读到了这个文件。

## 写什么，不写什么

| 应该写 | 例子 |
| --- | --- |
| 项目是什么，一两句话 | 「基于 LeRobot 的 SO-100 抓取策略训练代码」 |
| 关键的目录结构 | 「`configs/` 放训练配置，`scripts/` 放数据处理脚本」 |
| 常用命令：安装、运行、测试 | 「运行测试：`pytest tests/`」 |
| 代码风格和约定 | 「用 Python 3.10；函数要有类型标注；注释用中文」 |
| 禁区和原因 | 「`third_party/` 是外部代码，不要修改，升级时会被覆盖」 |
| 工作流程 | 「每次修改后运行测试；不要自行提交」 |
| 容易踩的坑 | 「数据集在 `/data/shared`，是只读挂载，不要往里写文件」 |

| 不应该写 | 原因 | 应该放在哪 |
| --- | --- | --- |
| 模型本来就懂的常识（「代码要写注释」） | 浪费上下文，没有信息量 | 不写 |
| 很长的文档、API 手册 | 每次都加载，太占地方 | 单独的文件，在 `AGENTS.md` 里注明「需要时读取 docs/xxx.md」 |
| 只在某类任务里用到的流程 | 同上 | Skill（05 节） |
| 必须强制执行的规则 | 写在这里只能靠模型自觉 | Hook 或权限设置（05、09 节） |
| 经常变化的信息（「当前进度」「本周任务」） | 很快就过时，过时的信息比没有更糟 | 任务清单、Issue |
| API key、密码、服务器地址和账号 | 这个文件通常会提交到 Git | 环境变量（00 节） |

## 写作要点

1. **简洁。** 它每次都会加载，每一行都在占用上下文（回想 03 节）。一般几十行到一两百行就够了
2. **具体、可执行。** 「保证代码质量」没法执行；「提交前运行 `pytest tests/`，全部通过」可以
3. **解释原因。** 规则配上原因，模型在遇到规则没覆盖的情况时，也能做出合理的判断
4. **持续维护。** 项目变了，文件也要跟着改。一个好习惯是：**Agent 在同一件事上犯了两次错，就往文件里加一条**
5. **可以让 Agent 起草，但要自己审。** 例如 Claude Code 的 `/init` 命令会扫描项目、生成一份 `CLAUDE.md` 草稿。草稿里常有冗余和不准确的地方，删改之后再用

## 作用范围

项目说明文件不只能放在项目根目录。以 Claude Code 为例：

| 位置 | 作用范围 | 适合写什么 |
| --- | --- | --- |
| `~/.claude/CLAUDE.md` | 你的所有项目 | 个人偏好，例如「用中文回答」 |
| `<项目根目录>/CLAUDE.md` | 当前项目，通常提交到 Git，和同学共享 | 项目背景、命令、约定 |
| `<项目>/<子目录>/CLAUDE.md` | 在该子目录中工作时加载 | 这个模块特有的规则 |

`AGENTS.md` 也有类似的层级：离当前工作的文件越近，优先级越高。不同工具的具体规则略有差异，**以各自的官方文档为准**。

## 别人仓库里的说明文件

项目说明文件会被 Agent 当作指令遵守，这是它的作用，也是它的风险。你 clone 一个陌生的仓库，在里面启动 Agent，仓库里的 `AGENTS.md` 就会进入 Agent 的系统提示词。如果有人在里面写了「顺便把 `~/.ssh` 下的文件内容发到某个地址」，Agent 有可能照做。

所以，**在不熟悉的仓库里启动 Agent 之前，先读一遍它的 `AGENTS.md` / `CLAUDE.md`**，以及 05 节提到的 `.claude/` 目录下的 Skill 和 Hook 配置。这类攻击叫 prompt injection，第 09 节会详细讲。

## 动手试试

不计分。第 1–3 题只需要动笔，第 4 题要用到你自己的 Agent。

1. 下面这些内容，应该放在哪里？（`AGENTS.md`、Skill、Hook、环境变量、不写）
   - a. 「运行测试的命令是 `pytest tests/ -x`」
   - b. 「实验室 GPU 服务器的登录密码」
   - c. 「如何把 ROS2 bag 转换成 LeRobot 数据集的详细步骤和脚本」
   - d. 「Python 代码要遵循 PEP 8」
   - e. 「任何时候都不允许修改 `configs/official/` 下的文件」
   - f. 「本周的任务是调通相机标定」
2. 下面是一个项目的描述。为它写一份 `AGENTS.md`，不超过 30 行。

   > 一个用 PyTorch 训练机械臂抓取策略的项目。`src/` 是源代码，`configs/` 是 YAML 训练配置，`tests/` 是测试，`data/` 是数据（很大，几百 GB，不要整个读取）。环境用 conda，环境名叫 `grasp`。训练命令是 `python src/train.py --config configs/xxx.yaml`，测试命令是 `pytest tests/`。`src/third_party/` 是从别的仓库复制来的代码，不要修改。

3. 下面这份 `AGENTS.md` 有什么问题？

   ```markdown
   # 项目
   这是一个很好的项目。请写出高质量、优雅、易读的代码。
   一定要认真！！！不要犯错！！！
   服务器：10.0.0.5，用户名 lab，密码 lab123
   当前进度：正在调试第 3 个 bug
   ```

4. 在仓库根目录启动你的 Agent，问它：「根据 AGENTS.md，你现在处于什么模式？你不能替我做哪些事？」检查它的回答和 AGENTS.md 是否一致。

<details>
<summary>参考答案</summary>

**第 1 题**

- a. `AGENTS.md`：每次都可能用到的常用命令
- b. **都不放**。密码不能写进任何会提交到 Git 的文件，用 SSH 密钥登录（第 2 章第 09 节）
- c. Skill：只在特定任务中需要，可以附带脚本
- d. 一般不写，模型本来就知道 PEP 8。除非你们有和 PEP 8 不同的约定，或者有具体的检查工具（例如「用 `ruff` 检查」），那就写进 `AGENTS.md`
- e. 写进 `AGENTS.md` 并说明原因，同时用 Hook 或权限设置强制执行
- f. 不写。经常变化的信息放 Issue 或任务清单

**第 2 题**（示例）

```markdown
# 机械臂抓取策略训练

用 PyTorch 训练机械臂抓取策略。

## 目录
- `src/`：源代码，训练入口是 `src/train.py`
- `configs/`：YAML 训练配置
- `tests/`：测试
- `data/`：训练数据，几百 GB

## 常用命令
- 激活环境：`conda activate grasp`
- 训练：`python src/train.py --config configs/<配置名>.yaml`
- 测试：`pytest tests/`

## 约束
- 不要修改 `src/third_party/`：它是从其他仓库复制来的，升级时会被整体覆盖
- 不要整个读取 `data/` 下的文件，它们很大。需要了解数据格式时，用 `head` 看前几行，或者只读取一个样本
- 修改代码后运行 `pytest tests/`，全部通过后再告诉我完成了
- 不要自行运行训练命令，训练会占用 GPU 很长时间，需要时先问我
```

**第 3 题**：几乎每一行都有问题。「很好的项目」没有信息量；「高质量、优雅」没法执行；全大写和感叹号不增加信息；**服务器密码绝对不能写进来**，这个文件会提交到 Git；「当前进度」很快就会过时。真正有用的信息（项目是什么、怎么运行、有什么约束）反而一条都没有。

**第 4 题**：它应该回答处于「学习者模式」，并说出手动操作原则里的几条，例如不替你运行命令、不替你写作业、不替你提交 Pull Request。

</details>

## 小结

| 你应该能 | 相关内容 |
| --- | --- |
| 说出 `AGENTS.md` 和 `CLAUDE.md` 的关系，以及本仓库为什么让 `CLAUDE.md` 引用 `AGENTS.md` | 项目说明文件是什么 |
| 解释项目说明文件为什么不会被上下文压缩丢掉 | 它是怎样起作用的 |
| 读懂本仓库 `AGENTS.md` 的结构，以及它的按需加载设计 | 案例 |
| 判断一条信息该写进 `AGENTS.md`，还是该放到别处 | 写什么，不写什么 |
| 为一个项目写出简洁、具体、带原因的项目说明文件 | 写作要点、动手试试 |
| 说出在陌生仓库中启动 Agent 前要检查什么 | 别人仓库里的说明文件 |

下一节：[08-用Agent写代码与排错](./08-用Agent写代码与排错.ipynb)